# LoRA: Low-Rank Adaptation of Large Language Models
### Annotated Implementation

**Hu et al., 2021 · [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)**  
Edward Hu, Yelong Shen, Phillip Wallis, Zeyuan Allen-Zhu, Yuanzhi Li, Shean Wang, Lu Wang, Weizhu Chen · Microsoft Corporation

---

This notebook is an annotated walkthrough of the LoRA paper, in the style of the
[Harvard Annotated Transformer](https://nlp.seas.harvard.edu/2018/04/03/attention.html).


### Background

Many applications in modern natural language processing need one massive general trained model to adapt a diverse class of problems downstream. Such adaptations are done by fine-tuning. Which is the process that updates all the pre-trained model parameters specifically for the new task. The biggest problem is fine-tuning trains the same amount of parameters as the original mode, which leads to a serious amount of cost with each fine-tune for different applications. As models are getting larger and larger with more parameters and tokens, this problem increases to a critical point hard to ignore. 

There were multiple different mitigations for a solution but they often introduced inference latency. Also, these methods fail to cover the fine-tuning baseline leading to a trade-off between quality and productiveness.

### Paper's Solution

They hypothize most models are over-parametrized, so they reside on a low intrinsic dimension. Low intrinsic meaning that even though LLM's have huge weight matrices, the actual meaningful changes during training or fine-tuning lie in a much lower-dimensional space. And the paper could capture this meaning lying in lower dimension with optimizing rank decomposition matrices(A and B) of some selected layers, while keeping the original training weights frozen/untouched. There are several advantages to this strategy: 

• An already pre-trained model can be shared with different applications for 
different tasks, only changing its A and B matrices, thus reducing the 
storage requirement and task-switching over-head significantly.

• This makes training  efficient and lowers the hardware requirements by up to 3
times when using adaptive optimizers since they only optimize the injected,
much smaller low-rank matrices.

• The simple linear design allows to merge the trainable matrices with the frozen weights, 
thus leading to no inference latency unlike other methods.

• LoRA could be combined with many of the other methods, such
as prefix-tuning. 

In [4]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset

DEVICE = "cpu"
print("PyTorch:", torch.__version__, "| Device:", DEVICE)

PyTorch: 2.8.0+cpu | Device: cpu


## Problem Statement

### Full fine-tuning (Equation 1)

$$\max_{\Phi} \sum_{(x,y)\in\mathcal{Z}} \sum_{t=1}^{|y|} \log P_{\Phi}(y_t \mid x,\, y_{<t})$$

| Symbol | Meaning |
|--------|---------|
| $\Phi$ | **All** model parameters (175B for GPT-3). The full thing is updated. |
| $(x,y) \in \mathcal{Z}$ | A (context, target) pair. For NL→SQL: $x$ = question, $y$ = SQL. For summarisation: $x$ = article, $y$ = summary. |
| $y_t$ | The $t$-th target token. |
| $y_{<t}$ | All target tokens before position $t$ — autoregressive conditioning. |
| $\log P_\Phi$ | Log-probability of the correct token. Maximising = minimising cross-entropy. |

### LoRA objective (Equation 2)

$$\max_{\Theta} \sum_{(x,y)\in\mathcal{Z}} \sum_{t=1}^{|y|} \log p_{\Phi_0 + \Delta\Phi(\Theta)}(y_t \mid x,\, y_{<t})$$

| Symbol | Meaning |
|--------|---------|
| $\Theta$ | The **small** LoRA parameters. For GPT-3, $|\Theta| \approx 0.01\%$ of $|\Phi_0|$. |
| $\Phi_0$ | Frozen pre-trained weights — shared across all tasks, never updated. |
| $\Delta\Phi(\Theta)$ | The update, now expressed via a low-rank factorisation. Same shape as $\Phi_0$, determined by far fewer free numbers. |

> **Key:** The *loss function* is identical to full fine-tuning. Only the *parametrisation*
> of the update changes — a representation constraint, not a change to the objective.

In [ ]:
class LoraLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int,
                 r: int = 4, alpha: float = 1.0, dropout: float = 0.0):
        super().__init__()
        self.r     = r
        self.scale = alpha / r 

        self.weight = nn.Parameter(torch.empty(out_features,in_features), requires_grad=False)

        self.lora_A = nn.Parameter(torch.empty(self.r, in_features))
        self.lora_B = nn.Parameter(torch.empty(out_features, self.r))

        self.dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()
        self._reset_parameters()

    def _reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = F.linear(x, self.weight)
        lora = F.linear(F.linear(self.dropout(x), self.lora_a),self.lora_B) * self.scale
        return base + lora

    def mergeweights(self) -> None:
        with torch.no_grad():
            self.weight.data += (self.lora_B @ self.lora_A) * self.scale


    def extra_repr(self):
        return (f"in={self.weight.shape[1]}, out={self.weight.shape[0]}, "
            f"r={self.r}, scale={self.scale:.3f}")

In [8]:
d = load_dataset('tiny_shakespeare', split="train[:200]")('train')
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
tok = tokenizer(d,"input_ids")
print(tok)



README.md: 0.00B [00:00, ?B/s]

tiny_shakespeare.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found tiny_shakespeare.py